In [1]:
import spacy
from transformers import pipeline, AutoTokenizer
from tqdm import tqdm
import os

/Users/charlesherr/venvs/venv1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = "data/medias_txt/Axios.txt"

In [3]:
nlp = spacy.load(
    "en_core_web_sm", # loads the small English model
    disable=["ner", "parser", "tagger", "lemmatizer"] # disable unnecessary components
)
nlp.add_pipe("sentencizer") # add sentence boundary detection component

In [4]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_model = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    batch_size=16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# we do not use the roberta.pooler layer for sentient analysis so the error is safe to ignore

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [5]:
def read_in_chunks(file, chunk_size=512*512):
    while True:
        data = file.read(chunk_size)
        if not data:
            break
        yield data

In [6]:
# read a txt file in chunks and yield sentences

def sentence_stream_from_txt(path, chunk_size=512*512, min_len=20):
    buffer = ""

    with open(path, "r", encoding="utf-8") as f:
        for chunk in read_in_chunks(f, chunk_size):
            buffer += chunk.replace("\n", " ")

            doc = nlp(buffer)
            sentences = list(doc.sents)

            for sent in sentences[:-1]:  # we put aside the last sentence to avoid cutting it
                text = sent.text.strip()
                if len(text) >= min_len:
                    yield text

            buffer = sentences[-1].text

        if buffer.strip():
            yield buffer.strip()

In [7]:
# make chunks of sentences with a maximum number of tokens

def chunk_sentences_by_tokens(sentence_stream, max_tokens=500):
    chunk = []
    token_count = 0

    for sentence in sentence_stream:
        tokens = tokenizer.tokenize(sentence)
        n_tokens = len(tokens)

        # Si une phrase est trop longue, la tronquer à max_tokens
        if n_tokens > max_tokens:
            truncated = tokenizer.convert_tokens_to_string(tokens[:max_tokens])
            if chunk:
                yield " ".join(chunk)
                chunk = []
                token_count = 0
            yield truncated
            continue

        # Si ajouter cette phrase dépasse max_tokens
        if token_count + n_tokens > max_tokens and chunk:
            yield " ".join(chunk)
            chunk = []
            token_count = 0

        chunk.append(sentence)
        token_count += n_tokens

    if chunk:
        yield " ".join(chunk)


In [8]:
# sentiment analysis on a stream of batch_size text blocks
# the model receives batches of blocks with each block being max_tokens long

def sentiment_stream(text_block_stream, batch_size=16):
    batch = []

    for block in text_block_stream:
        batch.append(block)

        if len(batch) == batch_size:
            for result in sentiment_model(batch):
                yield result
            batch = []

    if batch:
        for result in sentiment_model(batch):
            yield result


In [9]:
def run_pipeline(path):
    sentences = sentence_stream_from_txt(path)
    chunks = chunk_sentences_by_tokens(sentences)
    sentiments = sentiment_stream(chunks)

    for sentiment in sentiments:
        yield sentiment

In [10]:
def label_to_score(label, prob):
    """
    Convertit un label en score numérique [-1, 1]
    positive → +prob
    negative → -prob
    neutral  → 0
    """
    if label == "positive":
        return prob
    elif label == "negative":
        return -prob
    return 0.0


In [11]:
def analyze_corpus(path, max_blocks=1_000):
    """
    Analyse le corpus de texte et renvoie :
    - counts : nombre de blocs par sentiment
    - proportions : proportion de chaque sentiment
    - score_mean : score moyen global [-1,1]
    
    max_blocks : nombre maximum de blocs à analyser (None = tout analyser)
    """
    sentences = sentence_stream_from_txt(path)
    chunks = chunk_sentences_by_tokens(sentences)
    results = sentiment_stream(chunks)

    # Initialisation
    counts = {"positive": 0, "neutral": 0, "negative": 0}
    score_sum = 0.0
    n_blocks = 0

    # Barre de progression avec tqdm
    if max_blocks is not None:
        pbar = tqdm(total=max_blocks, desc="Processing blocks")
    else:
        pbar = tqdm(desc="Processing blocks")

    for res in results:
        label = res['label']
        prob = res['score']
        counts[label] += 1
        score_sum += label_to_score(label, prob)
        n_blocks += 1

        pbar.update(1)  # incrémente la barre

        # Arrêt si on a atteint la limite
        if max_blocks is not None and n_blocks >= max_blocks:
            break

    pbar.close()

    # Proportions
    proportions = {k: v / n_blocks for k, v in counts.items()}

    # Score moyen global
    score_mean = score_sum / n_blocks if n_blocks > 0 else 0.0

    return counts, proportions, score_mean


In [12]:
# counts, proportions, score_mean = analyze_corpus(path)

# print("Nombre de blocs par sentiment :", counts)
# print("Proportions :", proportions)
# print("Score moyen global :", score_mean)

In [13]:
files = [f for f in os.listdir("data/medias_txt/") if f.endswith(".txt")]
print(files)

['Refinery 29.txt', 'Vox.txt', 'Business Insider.txt', 'Gizmodo.txt', 'Buzzfeed News.txt', 'Economist.txt', 'Politico.txt', 'New Republic.txt', 'Axios.txt', 'Mashable.txt', 'The Hill.txt', 'The Verge.txt', 'CNBC.txt', 'The New York Times.txt', 'Reuters.txt', 'Vice.txt', 'Fox News.txt', 'Hyperallergic.txt', 'Wired.txt', 'People.txt', 'CNN.txt', 'Washington Post.txt', 'TMZ.txt', 'TechCrunch.txt', 'Vice News.txt', 'New Yorker.txt']


In [14]:
# utiliser le code sur chaque article du dossier data/medias_txt/

os.makedirs("results", exist_ok=True)

for file_name in ['Economist.txt', 'Politico.txt', 'New Republic.txt', 'Axios.txt', 'Mashable.txt', 'The Hill.txt', 'The Verge.txt', 'CNBC.txt', 'The New York Times.txt', 'Reuters.txt', 'Vice.txt', 'Fox News.txt', 'Hyperallergic.txt', 'Wired.txt', 'People.txt', 'CNN.txt', 'Washington Post.txt', 'TMZ.txt', 'TechCrunch.txt', 'Vice News.txt', 'New Yorker.txt']:

    media_name = file_name.replace(".txt", "")
    print(f"\nAnalyse de {media_name}...")

    counts, proportions, score_mean = analyze_corpus(f"data/medias_txt/{file_name}")

    # Ajouter au fichier existant (mode "a" = append)
    with open("results/sentiment_results.txt", "a", encoding="utf-8") as output_file:
        output_file.write(f"Média: {media_name}\n")
        output_file.write(f"  Nombre de blocs par sentiment: {counts}\n")
        output_file.write(f"  Proportions:\n")
        output_file.write(f"    - Positive: {proportions['positive']:.2%}\n")
        output_file.write(f"    - Neutral:  {proportions['neutral']:.2%}\n")
        output_file.write(f"    - Negative: {proportions['negative']:.2%}\n")
        output_file.write(f"  Score moyen global: {score_mean:.4f}\n")
        output_file.write("-" * 80 + "\n\n")

    print(f"\n✅ {media_name} ajouté à results/sentiment_results.txt")


Analyse de Economist...


Processing blocks:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing blocks: 100%|██████████| 1000/1000 [01:22<00:00, 12.17it/s]



✅ Economist ajouté à results/sentiment_results.txt

Analyse de Politico...


Processing blocks: 100%|██████████| 1000/1000 [01:44<00:00,  9.52it/s]


...


✅ TechCrunch ajouté à results/sentiment_results.txt

Analyse de Vice News...


Processing blocks: 100%|██████████| 1000/1000 [01:44<00:00,  9.61it/s]



✅ Vice News ajouté à results/sentiment_results.txt

Analyse de New Yorker...


Processing blocks: 100%|██████████| 1000/1000 [01:45<00:00,  9.44it/s]


✅ New Yorker ajouté à results/sentiment_results.txt


In [ ]:
# Générateur de phrases ou blocs
sentences = sentence_stream_from_txt(path)
chunks = chunk_sentences_by_tokens(sentences)
results = sentiment_stream(chunks)

# Nombre d’exemples à afficher
n_examples = 5
pos_examples = 0
neg_examples = 0

for chunk, res in zip(chunks, results):
    if res['label'] == 'positive' and pos_examples < n_examples:
        print("POSITIVE BLOCK:", chunk)
        print("Score:", res['score'])
        print("---")
        pos_examples += 1
    elif res['label'] == 'negative' and neg_examples < n_examples:
        print("NEGATIVE BLOCK:", chunk)
        print("Score:", res['score'])
        print("---")
        neg_examples += 1
    
    if pos_examples >= n_examples and neg_examples >= n_examples:
        break


NEGATIVE BLOCK: Among the memorable passages: "He looked me squarely in the eyes and, with a hint of a smile, said: 'Roger, I want to take the next step. I want to see if Donald Trump can win the White House. Is this country ready for President Trump? The one thing I do know is that I'm better than any of those assholes who are running.'" " Drudge led the charge, posting a top center headline and photograph on June 16, 2015, the day Trump declared his candidacy, proclaiming 'Donald Goes for White House.'" " On the Friday before Easter [2016], Trump called me at my south Florida home. ' Can they really steal this thing from me?' ... What should I do?' ' Call my former partner, Paul Manafort.' ... Trump asked for Manafort's cell phone number and I provided it." "[
Score: 0.5687109231948853
---
POSITIVE BLOCK: That amount will bring the firm's total price tag for resolving this scandal to $19.2 billion, more than the firm had predicted it would have to shell out. What's next: The deal sti

In [16]:
# estimation du nombre d'itérations possibles pour l'algorithme

with open(path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Nombre de caractères : {len(text):,}")

n_caracters = 1_000_000 #len(text)
n_tokens = n_caracters // 4  # approx.
n_iterations = n_tokens / 200

print(f"nombre maximal de blocks de 200 tokens qu'on peut faire avec un texte de {n_caracters} caractères : {n_iterations}")

Nombre de caractères : 67,450,179
nombre maximal de blocks de 200 tokens qu'on peut faire avec un texte de 1000000 caractères : 1250.0
